# Graph & Data Utilities

> Provide access easier access to functions to help acquire, prepare, and use graphs and genetic data.  

In [ ]:
#| default_exp util

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import io
import os, json, re
import requests
import numpy as np
import pandas as pd
import torch

## Reading marker data

In [ ]:
#| export
def read_vcf(
        path:str # File path 
        )->pd.DataFrame:
    "source: https://stackoverflow.com/questions/70219758/vcf-data-to-pandas-dataframe"
    with open(path, 'r') as f:
        lines = [l for l in f if not l.startswith('##')]
    return pd.read_csv(
        io.StringIO(''.join(lines)),
        dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str,
            'QUAL': str, 'FILTER': str, 'INFO': str},
        sep='\t'
    ).rename(columns={'#CHROM': 'CHROM'})

In [ ]:
#| export
def  vcf_table_to_matrix(
        vcf:pd.DataFrame # dataframe from `read_vcf`
        ) -> np.array:
    "Take a dataframe containing a vcf table and return a 3d array of shape (snps, genotypes, nucleotide) with nucleotides ordered ACGT."
    # We begin by finding all the genotypes in the VCF. These will be all the non-standard columns.
    genotypes = [e for e in list(vcf) if e not in ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'FORMAT']]

    # Next we use pandas to split the data columns into separate columns for each copy. Missing ('.') values are replaced with '4'. These values will act as an index.
    _ = vcf.loc[:, genotypes]

    _ = pd.concat([_.loc[:, genotype].str.replace('.', '4'
                                    ).str.split('/', expand=True).add_prefix(genotype+'_'
                                    )
                for genotype in genotypes], axis= 1)


    # The result of a nucleotide index (e.g. 0) changes with each row but we can expand the `REF` and `ALT` columns into their own table. Now the values in a row of table `_` corresponds to a column in this table (`tmp_ref`) which contains the correct nucleotide. (e.g. in `_` Row 0, Col B14A/H95_0 -> 0 in `tmp_ref` Row 0, Col 0 -> C)

    tmp_ref = (vcf['REF']+',' + vcf['ALT']).tolist()
    # get rid of '.'. Since I'm adding counts and '.' equates to no snp (all 0s) this will have no effect.
    tmp_ref = [e.replace(',', '').replace('.', '') for e in tmp_ref]
    tmp_ref = [[ee for ee in e]+['.' for j in range(5-len(e))] for e in tmp_ref]
    tmp_ref = pd.DataFrame(tmp_ref)

    # With this set, we can iterate over all the columns in `_` replacing all the rows with the corresponding values `tmp_ref`.
    # using the temp lookup go over each column and convert the index into the appropriate nucleotide
    for e in list(_):
        for i in range(5):
            # print(e, i)
            _.loc[(_.loc[:, e]==str(i)), e] = tmp_ref[i]

    # Next the columns for each genotype get recombined so that "C", "C" -> "CC".
    _ = pd.concat([pd.DataFrame(_[f'{genotype}_0']+_[f'{genotype}_1'], columns=[genotype]) for genotype in genotypes], axis=1)


    # We'll initialize the 3d array to hold the probabilites of seeing each nucleotide at each position and genotype. We'll also convert the dataframe `_` to a matrix. 
    # to hold probabilities
    acgt = np.zeros((
        _.shape[0],
        _.shape[1],
        4           # nucleotides
        ))
    _ = _.to_numpy()


    # Now we create a mapping from characters to probabilies. Beginning with a mapping for individual nucleotides to counts we use a dictionary comprehension to expand this to all pariwise combinations. Finally the counts are normalized to probabilites.
    # expand dictionary into all pairwise combinations and their probability tensor
    acgt_to_count = {
        'A': np.array([1, 0, 0, 0]),
        'C': np.array([0, 1, 0, 0]),
        'G': np.array([0, 0, 1, 0]),
        'T': np.array([0, 0, 0, 1]),
        '.': np.array([0, 0, 0, 0])
        }

    acgt_to_pr = {k1+k2:(acgt_to_count[k1]+acgt_to_count[k2]) 
                for k1 in acgt_to_count 
                for k2 in acgt_to_count}

    for k in acgt_to_pr:
        if acgt_to_pr[k].sum() > 0. :
            acgt_to_pr[k] = (acgt_to_pr[k]/acgt_to_pr[k].sum())


    # Now we can iterate over the keys in this dictionary ('AA' to '..'), find all the matches in `_` and then fill in the appropriate probabilities in `acgt`.
    for k in acgt_to_pr:
        acgt[(_ == k)] = acgt_to_pr[k]

    return acgt

In [ ]:
#| export
def  hmp_table_to_matrix(
        hmp:pd.DataFrame # hapmap read in as a dataframe
        ) -> np.array:
    "Take a dataframe containing a and return a 3d array of shape (snps, genotypes, nucleotide) with nucleotides ordered ACGT."

    # We begin by finding all the genotypes in the VCF. These will be all the non-standard columns.
    genotypes = [e for e in list(hmp) if e not in ['rs#', 'alleles', 'chrom', 'pos', 'strand', 'assembly#', 'center', 'protLSID', 'assayLSID', 'panelLSID', 'QCcode' ]]
                                                # 'CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'FORMAT'

    # relative to processing vcfs, we can skip over a lot of work because the nucleotides are at each site are already provided to us. 
    hmp_mat = hmp.loc[:, genotypes].to_numpy()
    hmp_mat

    # hapmap contains 'N' for 'any'. 
    # I'm recodeing this as '.' so that I can use the vcf processing code. 
    hmp_mat[hmp_mat == 'N'] = '.'

    acgt = np.zeros((
        hmp_mat.shape[0],
        hmp_mat.shape[1],
        4           # nucleotides
        ))

    acgt_to_count = {
        'A': np.array([1, 0, 0, 0]),
        'C': np.array([0, 1, 0, 0]),
        'G': np.array([0, 0, 1, 0]),
        'T': np.array([0, 0, 0, 1]),
        '.': np.array([0, 0, 0, 0])
        }

    for k in acgt_to_count.keys():
        acgt[hmp_mat == k] = acgt_to_count[k]

    return acgt




## Reading Gene Annotation files

In [ ]:
#| export
def _read_gene_annotation_table(
        filepath:str = './genomic.gff', # path to a gene annotation gff, gtf, or gff3 
        as_gff3:bool = True # use gff3 field names or infer based on extension ((gff, gtf) or (gff3))
        )->pd.DataFrame:
    "Read a gene annotation file. Return all rows with tabs as a table"
    # use fields based on the file path
    extension = 'gff3' if as_gff3 else filepath.split('.')[-1]

    match extension:
        case 'gff' | 'gtf':
            # https://useast.ensembl.org/info/website/upload/gff.html
            fields = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']
        case 'gff3':
            # https://plastid.readthedocs.io/en/latest/concepts/gff3.html
            # "The special GFF3 line ###, which indicates that Parent-child relationships for preceding features have been fully resolved."
            fields = ['seqid',   'source',   'type', 'start', 'end', 'score', 'strand', 'phase', 'attributes']

    with open(filepath, 'r') as f:
        dat = ''.join(f.readlines()).split('\n') # get rid of trailing newlines
        
    dat = [dat[i] for i in range(len(dat)) if re.findall('\t', dat[i])]
    dat = pd.DataFrame([e.split('\t') for e in dat], columns=fields)
    return dat


In [ ]:
#| export
def _gene_annotation_table_expand_attributes(
        table:pd.DataFrame # Table containing gene annotations from `_read_gene_annotation_table`
        )->pd.DataFrame:
    "Expand the 'attributes' field in a gene annotation table into many fields. "
    # earlier version basd on gff3
    # Parse attributes column into dictionary of dictionaries
    # e.g. row 0 "ID=chromosome:1" -> 0: {'chromosome': '1'},
    # def _parse_gff3_attributes(attributes_str):
    #     """Parse attributes column into dictionary. e.g. "ID=chromosome:1" -> {'chromosome': '1'}"""
    #     return {i:j for i,j in 
    #             # list of key,value pairs, sep is either : or =
    #             [(lambda x: x.split(':') if ':' in x else x.split('='))(e)
    #              # entry sep is ;
    #                for e in attributes_str[3:].split(';')]
    #                } 
    
    def _parse_gff_attributes(
            attributes_str:str # attributes from a row of a gene annotation file. Likely in a table field from `_read_gene_annotation_table`
            ):
        """Parse attributes column into dictionary. e.g. "ID=chromosome:1" -> {'chromosome': '1'}"""
        # check that each entry has 
        # list of key,value pairs, sep is either : or =
        x = [e.split('=') for e in attributes_str.split(';')]
        # confirm there are the right number of values to unpack
        vals = [e for e in x if len(e) != 2]
        if vals == []:
            return {i:j for i,j in x}
        else:
            print(attributes_str)
            print(vals)
            print(x)
            return {}
    # Here are two versions either processing each entry or processing the unique ones and merging them. 
    # Not merging seems to be slightly faster in a test. The difference is small ~13.7 vs 14.5 but we'll use the faster version

    # 13.8, 14.8, 14.9
    # res = pd.DataFrame([
    #  {'attributes':e} | _parse_gff_attributes(attributes_str = e) # | here is the dictonary merge opperator
    #  for e in list(set(table['attributes'].tolist()))
    #  ]
    #  )
    # table = table.merge(res)

    # 13.4, 13.8, 14.1
    table = pd.concat([
        table, 
        pd.DataFrame([_parse_gff_attributes(attributes_str = e) 
                      for e in table['attributes'].tolist()])
        ], 
        axis=1)
    table = table.drop(columns=['attributes']).drop_duplicates()
    return table


## Connection Table:

### Downloading and Transforming KEGG data:

In [ ]:
#| export
def _get_available_catalog(
        species:str = 'gmx', # KEGG species code
        cache:bool = True,   # Store/Use local copy?
        cache_dir:str = './' # Local storage path
        ) -> dict:           # Dict of {catalog entry : number}
    "Parse webpage to dict of {catalog entry : number}"
    cache_path = f'{cache_dir}{species}_catalog.json'
    dl = True
    if cache:
        if os.path.exists(cache_path):
            dl = False

    if not dl:
        with open(cache_path, 'r') as f: 
            res = json.load(f)

    elif dl:
        x = requests.get(f'https://www.genome.jp/kegg-bin/show_organism?menu_type=gene_catalogs&org={species}')
        x = x.text
        x = x.split('\n')
        x = [e for e in x if re.findall('/brite/.+keg', e)]
        catalog = {
            k:re.findall('\d+', v)[0]
            for v,k in 
            [e.split('href="/brite/')[-1].replace('</a><br>', '').split('>') for e in x]
        }
        res = x

        if cache:
            with open(cache_path, 'w') as f:
                json.dump(res, f)

    return res

In [ ]:
#| export
def _get_json(
        species:str = 'gmx', # KEGG species code
        catalog_num:str = '00001', # KEGG functional hierarcy number
        cache:bool = True,   # Store/Use local copy?
        cache_dir:str = './' # Local storage path
        ) -> dict:           # Dict of functional hierarcy json
    'Retrieve json of KEGG functional hierarchy'
    cache_path = f'{cache_dir}{species}{catalog_num}.json'
    dl = True
    if cache:
        if os.path.exists(cache_path):
            dl = False

    if not dl:
        with open(cache_path, 'r') as f: 
            res = json.load(f)

    elif dl:
        json_url = f'https://www.genome.jp/kegg-bin/download_htext?htext={species}{catalog_num}&format=json&filedir=kegg/brite/{species}'
        res = requests.get(url=json_url)
        res = res.json()
    
        if cache:
            with open(cache_path, 'w') as f:
                json.dump(res, f)

    return res

In [ ]:
#| export
def _peel(
        inp:dict | list # KEGG functional hierarcy, output from `_get_json`
        )->list:
    # Convert kegg json to src->tgt table
    "Convert KEGG functional hierarcy (`_get_json`) from a dict with dict/list children to a list of lists"
    if type(inp) == dict:
        if 'children' not in inp.keys():
            return inp['name']
        else:
            return [inp['name'], _peel(inp['children'])]
    if type(inp) == list:
        return [_peel(e) for e  in inp]

In [ ]:
#| export
def _mk_connections(
        tmp:list # KEGG functional hierarcy, as list of lists from `_peel`. Either full or part of it.
        )-> tuple:
    "Given a KEGG list of lists [A, [B, [C, D]]] return a list of the connections with the first item ([A, B] and the next item [B, [C, D]])"
    # Test if x is a list of atomics (strings not lists)
    islat = lambda x: ((type(x) == list) & 
                   ({str} == set([type(e) for e in x])))
    
    connections = []
    tmp_types = [type(e) for e in tmp]

    if tmp_types == [str, list]: 
        tmp1_types = [type(e) for e in tmp[1]]

        if tmp1_types == [str, list]: # one child, is a stem
            connections.append([tmp[0], tmp[1][0]])
        
        elif {str}  == set(tmp1_types): # all leaves
            for e in tmp[1]:
                connections.append([tmp[0], e])

        elif {list} == set(tmp1_types): # multiple children which might be stems or leaves
            for e in tmp[1]:
                e_types = [type(ee) for ee in e]
                if e_types == [str, list]:
                    connections.append([tmp[0], e[0]])
                elif set(e_types) == {str}:
                    connections.append([tmp[0], e])
                else:
                    print(f'what is {e}')
            # connections.append([tmp[0], tmp[1][0]])

            # are any lists lists of strings?
        elif {list, str} == set(tmp1_types):
            tmp1_leaves = [e for e in tmp[1] if type(e) == str]
            tmp1_lists  = [e for e in tmp[1] if type(e) == list]
            for e in tmp1_leaves:
                connections.append([tmp[0], e])
            
            # overwrite without the leaves
            tmp[1] = tmp1_leaves

        else:
            print(f'Entry\'s index 1 contains unexpected types {tmp1_types}')
    
    else:
        print(f'Entry is of unexpected types: {tmp_types}')

    return connections, tmp[1]

In [ ]:
#| export
def _connections_from_peeled_json(
        tmp:list, # KEGG functional hierarcy, as list of lists from `_peel`
        max_iter:int = 600, # Maximum attempts process all branches/stems/leaves in the queue
        print_queue_len:bool = False # Should current queue length be shown?
        )-> list:
    "Given a KEGG list of lists, find all the connections between nodes returning a list of [[src, tgt], [src, tgt], ...]"
    # Test if x is a list of atomics (strings not lists)
    islat = lambda x: ((type(x) == list) & 
                   ({str} == set([type(e) for e in x])))

    queue = [tmp]
    connections = []
    
    for iter in range(max_iter):
        if print_queue_len: print(f'{iter}\t{len(queue)}')
        if len(queue) == 0:
            break
        tmp = queue.pop()
        cxn, tmp = _mk_connections(tmp = tmp)

        connections.append(cxn)
        if set([type(e) for e in tmp]) == {list}:
            for e in tmp:
                queue.append(e)
        elif islat(tmp):
            pass
        elif ([str, list] == [type(e) for e in tmp]):
            queue.append(tmp)
        else:
            print('queue elememt is neither list of atomics nor list of lists')
            break
        # todo if its [str, list] then add it as a list. Don't separate out entries

        if iter == (max_iter-1):
            print('`max_iter` reached! Returning None. Rerun with higher `max_iter`.')
            connections = None
        
    return sum(connections, [])


In [ ]:
#| export
def _get_kegg2ncbi(
        species:str = 'gmx', # KEGG species code
        cache:bool = True,   # Store/Use local copy?
        cache_dir:str = './' # Local storage path
        )->dict:
    "Return a dict mapping kegg ids to ncbi ids."
    cache_path = f'{cache_dir}{species}_kegg2ncbi.json'
    dl = True
    if cache:
        if os.path.exists(cache_path):
            dl = False

    if not dl:
        with open(cache_path, 'r') as f: 
            res = json.load(f)

    elif dl:
        x = requests.get(f'https://rest.kegg.jp/conv/ncbi-geneid/{species}')
        x = [e.split('\t') for e in x.text.split('\n')]
        x = [e for e in x if (
            (e != ['']) & 
            (e == e))
        ]

        x = {k:v for k,v in x}
        res = x
    
        if cache:
            with open(cache_path, 'w') as f:
                json.dump(res, f)

    return res

### Switching between graph representations

For easy conversion between different representations of the connections in the graph we have these six functions. This is more than is strictly necessary since we could convert (e.g.) a dictionary to a df using (tensor, node) as an intermediate. Any future representations should be converted with this strategy rather than providing exhausive a<-->b functions. 

```     
    (tensor, nodes)
 ^ ^               ^ ^
 | |               | |
d_to_t           df_to_t
t_to_d           t_to_df
 | |               | |
 v v               v v
dict <- d_to_df -> df
     <- df_to_d ->
```

For convenience these are wrapped in `convert_connections`. Using a wrapper allows future functionality to be added with out the user needing to worry about how the representations are converted. 



In [ ]:
#| export
def d_to_df(inp:dict)-> pd.DataFrame:
    out = pd.DataFrame(
        sum(
            # iterate over all keys and values in the list of values for each key. 
            # return lists of src->tgt tuples, collapse from list of lists to list
            # convert to df.
            [[(e, ee) for ee in inp[e]]
             for e in inp],
             []
             ), 
             columns=['src', 'tgt']
             )
    return out

In [ ]:
#| export
def df_to_d(inp:pd.DataFrame)->dict:
    edge_dict = {}
    keys = list(set(inp.src.tolist()))
    for key in keys:
        edge_dict = edge_dict | {key: inp.loc[(inp.src == key), 'tgt'].tolist()}
    return edge_dict


In [ ]:
#| export
def d_to_t(inp:dict)->tuple:
    src   = list(inp.keys())
    tgt   = list(set(sum([inp[k] for k in src], [])))
    nodes = list(set(src+tgt)).copy()
    node2i= {nodes[i]:i for i in range(len(nodes))}

    out = torch.zeros(len(nodes), len(nodes)).to(int)

    for key in src:
        for key2 in inp[key]:
            out[node2i[key], node2i[key2]] = 1

    return out, nodes


In [ ]:
#| export
def t_to_d(inp:torch.tensor, nodes:list)->dict:
    idxs = torch.linspace(0, len(nodes)-1, len(nodes)).to(int)
    not0 = inp != 0

    out = {}
    for i in range(len(nodes)):
        node = nodes[i]
        tgt_idxs = ((inp[i, ] * idxs)[not0[i, ]]).tolist()
        out = out | {node: [nodes[i] for i in tgt_idxs] }
    # prune keys with empty lists
    out = {
        k:v for k,v in 
        [(k, out[k]) for k in out]
        if v != []
        }
    return out


In [ ]:
#| export
def df_to_t(inp:pd.DataFrame)->tuple:    
    src   = list(set(inp.src.tolist()))
    tgt   = list(set(inp.tgt.tolist()))
    nodes = list(set(src+tgt)).copy()
    node2i= {nodes[i]:i for i in range(len(nodes))}

    out = torch.zeros(len(nodes), len(nodes)).to(int)
    for i in inp.index:
        key, key2 = inp.loc[i, 'src'], inp.loc[i, 'tgt']
        out[node2i[key], node2i[key2]] = 1

    return out, nodes

In [ ]:
#| export
def t_to_df(inp:torch.tensor, nodes:list)-> pd.DataFrame:
    idxs = torch.linspace(0, len(nodes)-1, len(nodes)).to(int)
    not0 = inp != 0

    out = []
    for i in range(len(nodes)):
        node = nodes[i]
        tgt_idxs = ((inp[i, ] * idxs)[not0[i, ]]).tolist()
        out.append([(node, nodes[i]) for i in tgt_idxs])

    out = pd.DataFrame(sum(out, []), columns=['src', 'tgt'])
    return out

Here we can demonstrate that the same graph structure is recovered as we move thorugh these forms:

In [ ]:
test1 = [{
    'A': ['In1'],
    'B': ['In1'],
    'C': ['In2'],
    'D': ['B', 'C'],
    'E': ['A'],
    'Out': ['C', 'D', 'E']
    }]

# go through each of the conversion functions clockwise to convert dict -> tensor, nodes -> df -> dict
test1.append(d_to_t( inp=test1[-1]))
test1.append(t_to_df(inp=test1[-1][0], nodes=test1[-1][1]))
test1.append(df_to_d(inp=test1[-1]))

def _compare_dict_unordered(d1, d2):
    key_bool = d1.keys() == d2.keys()
    keys = list(d1.keys())
    val_bool = True if False not in [
        sorted(d1[k]) == sorted(d2[k])
        for k in keys
        ] else False
    return key_bool, val_bool

# Check that the keys and values match (but can be differently ordered)
assert (True, True) == _compare_dict_unordered(d1 = test1[0], d2 = test1[-1])

test1.append(d_to_df(inp=test1[-1]))
test1.append(df_to_t(inp=test1[-1]))
test1.append(t_to_d( inp=test1[-1][0], nodes=test1[-1][1]))

assert (True, True) == _compare_dict_unordered(d1 = test1[0], d2 = test1[-1])



# assert gph_d.keys() == gph_dr2.keys()
# # same entries for each key?
# assert [] == [(k, gph_d[k], gph_dr2[k])
#               for k in 
#               gph_d.keys()
#               if not sorted(gph_d[k]) == sorted(gph_dr2[k])
#               ]


# now test the other direction



In [ ]:
#| export

def convert_connections(
        inp, # Graphs as a dict of {parent: [children]}, nxn tensor with 0/1 for connections, or a dataframe of src,targets
        to: str, # Output representation 'dict', 'tensor', 'DataFrame'
        node_names:list | None = None # list of the nodes represented by the columns/rows in a 0/1 tensor  
):
    inp_type = type(inp)
    to = to.lower()
    if (inp_type == torch.tensor) & (node_names == None):
        # if there's no node name list provided create one.
        node_names = [str(i) for i in range(inp.shape[0])]
    match (inp_type, to):
        case (dict, 'dataframe'):
            return d_to_df(inp=inp)
        case (dict, 'tensor'):
            return d_to_t( inp=inp)

        case (pd.DataFrame, 'dict'):
            return df_to_d(inp=inp)
        case (pd.DataFrame, 'tensor'):
            return df_to_t(inp=inp)
        
        case (torch.tensor, 'dict'):
            return t_to_d(inp=inp, nodes=node_names)            
        case (torch.tensor, 'dataframe'):
            return t_to_df(inp=inp, nodes=node_names)            
        case _:
            return None

In [ ]:
convert_connections(
        inp = {
    'A': ['In1'],
    'B': ['In1'],
    'C': ['In2'],
    'D': ['B', 'C'],
    'E': ['A'],
    'Out': ['C', 'D', 'E']
    }, 
    to = 'tensor',
node_names = None )

(tensor([[0, 0, 1, 0, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 1, 1, 0, 1],
         [1, 0, 0, 0, 0, 0, 0, 1],
         [0, 1, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 1, 0]]),
 ['B', 'A', 'In1', 'Out', 'D', 'E', 'In2', 'C'])

In [ ]:
inp = convert_connections(
        inp = {
    'A': ['In1'],
    'B': ['In1'],
    'C': ['In2'],
    'D': ['B', 'C'],
    'E': ['A'],
    'Out': ['C', 'D', 'E']
    }, 
    to = 'dataframe',
node_names = None )

In [ ]:
#| export

def order_connections(
        inp, # Representation of a graph -- dict, dataframe, or tensor.
        node_names = None # Optional, but should be used for tensor representations. Otherwise nodes will be numbered.
        ):
    "Given some connections, produce a dictionary with the distance of each node to the top node(s). {0:['out'] ... 3:['In1', 'In2']}"
    if type(inp) == pd.DataFrame:
        pass
    if type(inp) == dict:
        inp = convert_connections(
            inp = inp, 
            to = 'dataframe',
            node_names = None )
    if type(inp) == torch.tensor:
        if node_names == None:
            node_names = [str(i) for i in range(inp.shape[0])]
        inp = convert_connections(
            inp = inp, 
            to = 'dataframe',
            node_names = node_names)

    _ = inp.copy()
    out = {}

    src = list(set(_.src.tolist()))
    tgt = list(set(_.tgt.tolist()))
    inp_nodes = [e for e in tgt if e not in src].copy()

    for i in range(_.shape[0]*2):
        src = list(set(_.src.tolist()))
        tgt = list(set(_.tgt.tolist()))
        if src == []:
            break

        # find all src that are not targets
        top_nodes = [e for e in src if e not in tgt]
        out = out | {i: top_nodes}
        # remove the top nodes from consideration
        _ = _.loc[(~_.src.isin(top_nodes))]

    out = out | {max(list(out.keys()))+1: inp_nodes}
    return out

In [ ]:
_ = order_connections(inp = {
    'A': ['In1'],
    'B': ['In1'],
    'C': ['In2'],
    'D': ['B', 'C'],
    'E': ['A'],
    'Out': ['C', 'D', 'E']
    }, 
    node_names=None)

# if we want a sorted list:
_, sum([_[k] for k in _], [])

({0: ['Out'], 1: ['D', 'E'], 2: ['C', 'B', 'A'], 3: ['In1', 'In2']},
 ['Out', 'D', 'E', 'C', 'B', 'A', 'In1', 'In2'])

## Functions to streamline building VNNs

In [ ]:
#| export

def intersect_cxn_gff_nodes(gff, cxn, kegg2ncbi):
    # gff, cxn, kegg2ncbi -> gene_nodes_gff (?)gff w/ col
    species = list(kegg2ncbi.keys())[0].split(':')[0]

    nodes = list(set(cxn.src.tolist()+cxn.tgt.tolist()))
    # since the lamda deals with missing entries below we don't need to filter the inputs
    # gene_nodes = [e for e in nodes if re.match('\d+.*', e)]
    gene_nodes = nodes

    gene_nodes = pd.DataFrame(zip(
        gene_nodes,
        [f"{species}:{e.split(' ')[0]}" for e in gene_nodes]), columns=['cxn', 'kegg'])

    gene_nodes['ncbi']= [# lambda to deal with missing values
        (lambda x: kegg2ncbi[x] if x in kegg2ncbi.keys() else '')(e) 
        for e in gene_nodes.kegg.tolist()]

    # Transform Dbxref and join
    gff['ncbi'] = [f"ncbi-geneid:{e.split(':')[-1]}" for e in gff['Dbxref'].tolist()]

    gene_nodes_gff = gene_nodes.merge(gff, how='inner')
    gene_nodes_gff = gene_nodes_gff.sort_values(['chromosome', 'start', 'end']).reset_index(drop=True)

    print('\n'.join(
        ['Grouping\t| Count', 
        '--------\t| -----'
        ]+[f'{i}\t| {j}' for i,j in zip(
            ['All KEGG Nodes', 'GFF Nodes', 'Intersection'],
            [e.shape[0] for e in [gene_nodes, gff, gene_nodes_gff]]
            )]))

    return gene_nodes_gff

In [ ]:
#| export

def acgt_filter_taxa(acgt, acgt_taxa, shared_taxa):
    # acgt_filter_taxa acgt, acgt_taxa, shared_taxa -> acgt taxa2idx
    # Make sure the order of the taxa in acgt are as expected
    # currently dims are taxa, nucleotide, length
    print(f'acgt shape is {acgt.shape}. Confirm taxa is the 0th axis.')

    taxa2idx = {k:v for v,k in enumerate(acgt_taxa)}
    acgt = acgt[[taxa2idx[e] for e in shared_taxa], :, :]
    return(acgt, taxa2idx)



In [ ]:
#| export

def acgt_filter_snps(acgt, 
                     acgt_loci, 
                     gene_nodes_gff, 
                     include_adj = True):
    
    acgt_loci = acgt_loci.reset_index().rename(columns={'index':'acgt_l_idx'})
    # acgt_l_idx	chrom	pos
    # 	        0	1	24952
    #       	1	1	26003

    # acgt_loci.head()



    # A  B  C  D  E  F    <- snps sampled
    #    #######          <- geneic region
    # 
    # we return indexs for [A, B, C, D, E]
    # 
    # Whereas for gene:
    # A  B  C  D  E  F    <- snps sampled
    #      #              <- geneic region
    # 
    # we return indexs for [B, C] since there are not indexes within the gene
    #
    # and for gene:
    # A  B  C  D  E  F    <- snps sampled
    #                   # <- geneic region
    # 
    # we return indexs for [F] since there is no snp sampled at a higher position.

    # make sure we're working with ints
    acgt_loci.chrom = acgt_loci.chrom.astype(int)
    acgt_loci.pos   = acgt_loci.pos.astype(int)

    out = {}

    for i in gene_nodes_gff.index:
        node, chromosome, start, end = gene_nodes_gff.loc[i, ['cxn', 'chromosome', 'start', 'end']].tolist()
        node, chromosome, start, end = str(node), int(chromosome), int(start), int(end)
        # node, chromosome, start, end

        chrom_mask = (acgt_loci.chrom == chromosome)

        res = []
        if include_adj:
            # index that's closest but below the gene
            res += acgt_loci.loc[
                (chrom_mask & (acgt_loci.pos < start)), 
                ['acgt_l_idx']].max().tolist()

        # indexes within gene
        res += acgt_loci.loc[
            (chrom_mask & 
            ((acgt_loci.pos >= start) &
            (acgt_loci.pos <= end))
            ), 
            'acgt_l_idx'].tolist()

        if include_adj:
            # index that's closest but above the gene
            res += acgt_loci.loc[
                (chrom_mask & (acgt_loci.pos > end)), 
                ['acgt_l_idx']].min().tolist()
        
        # drop nans
        res = [e for e in res if e == e]
        out = out | {node:res}



    # Do we actually need the full set of genome snp indices or can we drop some?
    used_snps = sorted(list(set(sum([out[e] for e in out.keys()], []))))
    print(f'Using {len(used_snps)} of {acgt.shape[-1]} SNPs ({round(100*(len(used_snps) / acgt.shape[-1]), 3)}%)')



    # Reduce the snp dim of acgt and then update all the references in out. 
    idxorig2reduced = {k:i for i,k in enumerate(used_snps)}
    acgt = acgt[:, :, used_snps]

    out = {
        k:[idxorig2reduced[e] for e in v] 
        for k,v in 
        [(k, out[k]) for k in out.keys()]}

    inp_node_idx_dict = out.copy()

    # return a df of the position info for manhattan plots
    acgt_loci = acgt_loci.loc[used_snps, ['chrom', 'pos']].reset_index(drop=True).copy()
    return (acgt, inp_node_idx_dict, acgt_loci)

In [ ]:
#| export

def filter_connection_df(cxn, gene_nodes_gff):

    # Filter the input nodes to only those that we have a gene model for. 

    # This should be run multiple times until there are no nodes that are either
    # 1. Leaves that have no snps
    # 2. Branches without leaves that have no snps.

    # Instead of using a while loop, we iterate over the number of nodes. 
    print('Removing nodes without SNPs:')
    for itr in range(len(list(set(cxn.tgt+cxn.src)))):
        # The input nodes will be targets but not sources
        tgt =  list(set(cxn.tgt))
        src =  list(set(cxn.src))
        
        inp_nodes = [e for e in tgt if e not in src]
        # here are the input nodes that we don't have a gene model for.
        inp_nodes_prune = [e for e in inp_nodes if e not in gene_nodes_gff.cxn.tolist()]
        print(f'{itr}: {len(inp_nodes_prune)} of {len(inp_nodes)} ({round(100*(len(inp_nodes_prune)/len(inp_nodes)), 3)} %)')
        cxn = cxn.loc[~(cxn.tgt.isin(inp_nodes_prune)), ]
        if inp_nodes_prune == []:
            break

    cxn = cxn.reset_index(drop=True)
    return cxn

In [ ]:
#| export

def mk_vnnhelper(
        edge_dict,
        inp_tensor_lookup,
        num_nucleotides = 4, # this could also be 1 for major/minor allele. 
        params = {
            'default_out_nodes_inp'  : 1,
            'default_out_nodes_edge' : 1,
            'default_out_nodes_out'  : 1, #TODO set this based on the dimensions of y

            'default_drop_nodes_inp' : 0.0,
            'default_drop_nodes_edge': 0.0,
            'default_drop_nodes_out' : 0.0,

            'default_reps_nodes_inp' : 1,
            'default_reps_nodes_edge': 1,
            'default_reps_nodes_out' : 1,

            'default_decay_rate'     : 0
            }
            ):
    import sparsevnn.core   
    # older code assumes that a graph dictionary will contain leaves as keys with [] children. 
    # to accomodate this behavior we're going to 
    # 1. check if there are any nodes that are not keys and
    # 2. if there are spike them in. 
    all_nodes = list(set(sum([[k]+edge_dict[k] for k in edge_dict.keys()], [])))
    absent_nodes = {e:[] for e in all_nodes if e not in edge_dict.keys()}
    if absent_nodes != {}:
        edge_dict = edge_dict | absent_nodes
    # Now we don't need to worry about which structure the connection dict has

    myvnn = sparsevnn.core.VNNHelper(edge_dict = edge_dict)

    # We need to set attributes of the VNNHelper so the edges can be calculated.

    myvnn.set_node_props(
        key = 'inp', 
        node_val_zip = zip(
            myvnn.nodes_inp, 
            [len(inp_tensor_lookup[e])*num_nucleotides for e in myvnn.nodes_inp]
            ))

    myvnn.set_node_props(
        key = 'flatten', 
        node_val_zip = zip(myvnn.nodes_inp, [True for e in myvnn.nodes_inp]))

    for node_group in ['nodes_inp', 'nodes_edge', 'nodes_out']:
        for attr_type in ['out', 'drop', 'reps']:
            myvnn.set_node_props(
                key= attr_type, 
                node_val_zip=zip(
                    myvnn.__getattribute__(node_group),
                    [params[f'default_{attr_type}_{node_group}']   # repeat relevant default value
                    for e in myvnn.__getattribute__(node_group)]
                    # an example version of this is 
                    # myvnn.set_node_props(
                    #     key = 'reps', 
                    #     node_val_zip = zip(myvnn.nodes_inp, [default_reps_nodes_inp  for e in myvnn.nodes_inp]))
            ))


    # Scale node outputs by distance -----------------------------------------------
    dist = sparsevnn.core.vertex_from_end(
        edge_dict = myvnn.edge_dict,
        end =myvnn.dependancy_order[-1]
    )

    # overwrite node outputs with a size inversely proportional to distance from prediction node
    for query in list(dist.keys()):
        myvnn.node_props[query]['out'] = sparsevnn.core.dist_scale_function(
            out = myvnn.node_props[query]['out'],
            dist = dist[query],
            decay_rate = params['default_decay_rate'])
        
    # Expand out node replicates ---------------------------------------------------
    nodes = [node for node in myvnn.dependancy_order if myvnn.node_props[node]['reps'] > 1]

    node_expansion_dict = {
        node: [node if i==0 else f'{node}_{i}' for i in range(myvnn.node_props[node]['reps'])]
        for node in nodes}
    #   current       1st          2nd (new)      3rd (new)
    # {'100798274': ['100798274', '100798274_1', '100798274_2'], ...

    # the keys don't change here. The values will be updated and then new k:v will be inserted
    myvnn.edge_dict = {k:[e if e not in node_expansion_dict.keys() 
        else node_expansion_dict[e][-1]
        for e in myvnn.edge_dict[k] ] for k in myvnn.edge_dict}

    # now insert connectsion to new nodes: A -> A_rep_1 -> A_rep_2
    for node in node_expansion_dict:
        for pair in zip(node_expansion_dict[node][1:], node_expansion_dict[node]):
            myvnn.edge_dict[pair[0]] = [pair[1]]

    # now add those new nodes
    # create a new node for all the nodes
    for node in node_expansion_dict:
        for new_node in node_expansion_dict[node][1:]:
            myvnn.node_props[new_node] = {k:myvnn.node_props[node][k] for k in myvnn.node_props[node] if k != 'inp'}


    new_vnn = sparsevnn.core.VNNHelper(edge_dict= myvnn.edge_dict)
    new_vnn.node_props = myvnn.node_props
    myvnn = new_vnn

    # init edge node input size (propagate forward input/edge outpus)
    myvnn.calc_edge_inp()
    return myvnn


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()